In [0]:
%sql

SELECT *
FROM spotifyde_catalog.silver.user

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7544320532573464>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '\nSELECT *\nFROM spotifyde_catalog.silver.user\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activit

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS spotifyde_catalog

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS spotifyde_catalog.gold

In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotifyde_catalog.gold.dim_user_scd2
USING DELTA
LOCATION "abfss://gold@adlspotifyde.dfs.core.windows.net/scd2/dim_user_scd2"
AS
SELECT 
 row_number() over (order by user_id) AS DimUserKey_scd2,
 user_id,
 user_name,
 country,
 subscription_type,
 start_date,
 end_date,
 updated_at,

 current_date() AS effective_start_date,
 CAST (NULL AS DATE) as effective_end_date,
 TRUE AS is_current
 
 FROM spotifyde_catalog.silver.user;

In [0]:
%sql
SELECT * 
FROM spotifyde_catalog.gold.dim_user_scd2

In [0]:
%sql
-- Close the Old version
-- <=> handles NULL correctly.

MERGE INTO spotifyde_catalog.gold.dim_user_scd2 AS target
USING spotifyde_catalog.silver.user AS source

ON target.user_id = source.user_id

WHEN MATCHED
AND (
        target.user_name <> source.user_name 
    OR  target.country <> source.country 
    OR  target.subscription_type <> source.subscription_type 
    OR  target.start_date <> source.start_date 
    OR  target.end_date <> source.end_date 
    OR  target.updated_at <> source.updated_at 
)
AND target.effective_end_date IS NULL

THEN UPDATE SET 
target.effective_end_date  = current_date
target.is_current = FALSE




In [0]:
%sql
-- Handle New Version

INSERT INTO spotifyde_catalog.gold.dim_user_scd2(
 DimUserKey_scd2,
 user_id,
 user_name,
 country,
 subscription_type,
 start_date,
 end_date,
 updated_at,

 effective_start_date,
 effective_end_date,
 is_current
)
SELECT 
    row_number() over(order by user_id) 
    +
    ( select coalesce(max(DimUserKey_scd2),0)
      from spotifyde_catalog.gold.dim_user_scd2
    )
    AS DimUserkey_scd2,

    source.user_id,
    source.user_name,
    source.country,
    source.subscription_type,
    source.start_date,
    source.end_date,
    source.updated_at,

    current_date() AS effective_start_date,
    CAST(NULL AS DATE) AS effective_end_date,
    TRUE AS is_current

FROM spotifyde_catalog.silver.user as source

LEFT JOIN spotifyde_catalog.gold.dim_user_scd2 as target
  ON source.user_id = target.user_id
  AND target.is_current = TRUE

WHERE target.user_id IS NULL
